In [96]:
import numpy as np
from numpy.random import default_rng
rng = default_rng()     # common random number generator

# Assignment 2: $k$-Fold Cross-Validation

Implement a function that compares two learning algorithms using $k$-fold 
cross-validation. A target function is 
$f: \mathrm{R}^n \to \{0,1\}$, where $\mathrm{R}$ is the set of real numbers. The goal of a learning
algorithm is to identify a function $h: \mathrm{R}^n \to \{0,1\}$, called
hypothesis, from some class of functions $\mathcal{H}$ such that the
function $h$ is a good approximation of the target function $f$. The
only information the algorithm can use is a sample $S\subset X$ called
a training set, and the correct value $f(x)$, called label, for all $x \in S$.
The sample $S$ is a set of $n$ elements from $X$ randomly selected
according to some probabilistic distribution $D$.

Suppose that each learning algorithm is implemented as two functions `learn()` and `apply()`:
```
def learn(X, y, par):
    ...
    return learned_par
```
where

Param  |Meaning
------ |---------------------------------------------------------------
`learn`|is the name of the learning function,
`X`    |is a two-dimensional numpy array (X_traing vectors are the rows of `X`),
`y`    |is a vector of desired outputs (0/1), and
`par`  |is a variable containing parameters of the learning algorithm.

Such function returns learned parameters in the variable `learned_par`.

Further, we will need a function

```
def apply(learned_par, X):
    ...
    return out
```
where

Param  |Meaning
------ |---------------------------------------------------------------
`learned_par`|is a variable with the parameters learned by the function `learn`,
`X`    |is a two-dimensional numpy array (X_traing vectors are the rows of `X`),


that computes the learned function with parameters `learned_par` on input
vectors from a two-dimensional numpy array `X` (each row of the array is an input
vector). The returned value `out` is a vector of the results - zeros and ones.

Then it is possible to implement the following function.

In [97]:
def err(name_learn: str, name_apply: str, par: object, 
        X_train: np.ndarray, y_train: np.ndarray, 
        X_test: np.ndarray, y_test: np.ndarray) -> float:
    '''Trains the algorithm name_learn with parameters par
    on the train set X_train with the true labels y_train. Then,
    it compares predictions computed by the trained function on the test set
    X_test with the true labels y_test.
    It returns the error rate on the test set.
    
        error_rate = err(name_learn, name_apply, par, X_train, y_train, X_test, y_test)
        
    Args:
        name_learn: The name of the training algorithm.
        name_apply: The name of the function for applying the learned function.
        par: The parameters of the learning algorithm.
        X_train (2-d numpy array of floats): The training set; samples are rows.
        y_train (vector of integers 0/1): The true labels for the training samples.
        X_test (2-d numpy array of floats): The test set; samples are rows.
        y_test (vector of integers 0/1): The true labels for the test samples.
        
    Returns:
        error_rate: The error rate (a float between 0.0 and 1.0) of the function 
            with the name trained_apply on the test set X_test with correct 
            labels y_test after training using the function with the name
            name_learned with parameters par on the train set X_train with the correct
            labels y_train.
    '''
    trained = eval(name_learn+"(X_train,y_train,par)")  #training the algorithm on train data
    prediction = eval(name_apply+"(trained, X_test)")   #Computing predictions with the ttrained function on test data
    return (1/np.shape(y_test)[0])*np.sum(np.power(prediction - y_test,2))  #error of the prediction on the test data (MSE)

that computes the error of the function `name_learned` learned by the
algorithm `name_apply` with parameters `par` on a training set `X_train` and
desired outputs `y_train`. The error is computed on a test set `X_test` with the
desired outputs `y_test`. Note that the result of the function `err` is
always a real value from the closed interval $\langle0;1\rangle$, **not
the number of errors** made by the learned algorithm on the test set.
`name_learn` and `name_learned` are strings containing the names of the respective
functions. The respective functions can be evaluated using Python's function
`eval`, as shown below.

In [98]:
import math

fun = "math.log"
a = 16
b = 2
eval(fun+"(a, b)")

4.0

Then it is easy to implement the following function

In [99]:
def cross_val(name_learn1: str, name_apply1: str, par1: object, 
              name_learn2: str, name_apply2: str, par2: object, 
              X: np.ndarray, y: np.ndarray, k: int, shuffle: bool = True) -> (float, float):
    '''Computes the difference between errors and the standard deviation of the difference for
    two learning algorithms.
    
        delta, std = cross_val(name_learn1, name_apply1, par1, 
                               name_learn2, name_apply2, par2, 
                               X, y, k, shuffle)
                               
    Args:
        name_learn1: The name of the first training algorithm.
        name_apply1: The name of the function for applying the first learned function.
        par1: The parameters of the first learning algorithm.
        name_learn2: The name of the second training algorithm.
        name_apply2: The name of the function for applying the second learned function.
        par2: The parameters of the second learning algorithm.
        X (2-d numpy array of floats): The training set; samples are rows.
        y (vector of integers 0/1): The desired outputs for the training samples.
        k: The number of folds used in k-fold cross-validation.
        shuffle: If True, shuffle the samples into folds; otherwise, do not shuffle
                 the samples before splitting them into folds.
    
    Returns:
        delta: The estimated difference: error rate of the first algorithm minus 
            the error rate of the second algorithm computed using k-fold cross-validation.
        std: The standard deviation of the estimatied difference of errors.            
    '''
    (n,m) = np.shape(X)

    #Can k not divide n???? I suppose it always does because we are running it on matrix with dimension 600 wich is divisible by k=5 and k=6
    deltas = []

    #Permutation in case of Shuffle == True
    generator = np.random.RandomState(42)
    if shuffle == True: permutation = generator.permutation(n)
    else: permutation = range(n)

    batch_size = int(n/k)

    for i in range(0, n, batch_size):
        batch = permutation[i:i + batch_size]

        #Test data and target is the i-th k-fold
        test_data = X[batch] 
        test_target = y[batch]
        
        #Train data is all the k-folds except for the i-ths
        train_data = np.concatenate((X[permutation[0:i-1]], X[permutation[i+batch_size+1:n-1]]), axis=0)
        train_target = np.concatenate((y[permutation[0:i-1]], y[permutation[i+batch_size+1:n-1]]),axis = 0)

        error_algorithm1 = err(name_learn1, name_apply1, par1, train_data, train_target, test_data, test_target)
        error_algorithm2 = err(name_learn2, name_apply2, par2, train_data, train_target, test_data, test_target)
        
        deltas.append(error_algorithm1 - error_algorithm2)   #If positive, the 2nd algorithm is more accurate, if negative the 1st one.

    delta = (1/k)*np.sum(deltas)    #estimator for the difference between errors of the two algorithms
    std = np.sqrt( (1/(k*(k-1)))*np.sum([(deltas[i]-delta)**2 for i in range(k)]) )
    return (delta,std)

that estimates the difference between the errors of the hypothesis
`name_learned1` learned by a learning algorithm `name_learn1` with parameters `par1`
and the error of the hypotheses `name_learned2` learned by a learning algorithm
`name_learn2` with parameters `par2` using `k`-fold cross-validation on
patterns `X` with the desired outputs `y`. The function returns the estimated
difference of errors `delta` and estimated standard deviation `s` of this
estimator. **You should implement your own function using `numpy`, not use 
any implementation of cross-validation from any third-party library!**

If the value of `shuffle` is `False`, then
the order of samples must not be changed before partitioning into folds
for `k`-fold cross-validation, and all folds should be continuous parts
of `X`. If the value of `shuffle` is `True`, then the patterns from `X` 
should be assigned randomly into `k` folds.

*Note: The sizes of the folds can differ by at most 1.* 

For example, we can compare the errors of the perceptron learning
algorithm limited to at most 10 epochs and the perceptron learning
algorithm limited to at most 100 epochs. As the perceptron learning
algorithm, we will use the function `perc_learn` obtained using our implementation 
of class `Perceptron` from the lab. Similarly, for applying the learned perceptron
we will use `perc_apply` given below. For that, prepare a Python module `Perceptron` with the implementation of the class `Perceptron` from the previous lab. 

In [100]:
from Perceptron import Perceptron

def perc_learn(X: np.ndarray, y: np.ndarray, par: list) -> Perceptron:
    '''Train perceptron
    
        perc = perc_learn(X, y, par)
        
    Args:
        X (2-d numpy array of floats): The training set; samples are rows.
        y (vector of integers 0/1): The desired outputs for the training samples.
        par: List of the parameters for the perceptron learning algorithm
             [w, lr, max_epochs]:
                 w (nd.array): The extended weight vector of a perceptron.
                 lr (float): The learning rate of the perceptron.
                 max_epochs (int): The maximal number of learning epochs.
    
    Returns:
        perc (Perceptron): Trianed perceptron
    '''
    perc = Perceptron(init_weights=par[0])
    perc.fit(X,y, lr=par[1], max_epochs=par[2])
    return perc

def perc_apply(learned_par: Perceptron, X: np.ndarray) -> np.ndarray:
    '''Apply learned perceptron on inputs X.
    
        predictions = perc_apply(learned_par, X)
        
    Args:
        learned_par (Perceptron): An instance of the class Perceptron.
        X (2-d numpy array of floats): The training set; samples are rows.
        
    Returns:
        predictions (vector of 0/1): predictions of the perceptron learned_par 
                                     on the test set X.
    '''
    Out = learned_par.predict(X)
    return Out

Another learning algorithm we will implement is `memorizer`, which
memorizes all training samples and their true labels. Learned `memorizer` answers correctly on the inputs from the
training set and randomly otherwise:

In [101]:
def memorizer_learn(X: np.ndarray, y: np.ndarray, par: object = None) -> list:
    '''The learning algorithm that only remembers all training
    samples and the desired answers for them.

        learned_par = memorizer_learn(X, y, par)
    
    Args:
        X (2-d numpy array of floats): The training set; samples are rows.
        y (vector of integers 0/1): The desired outputs for samples from X.
        par: It is not used here, it is present here for compatility only
    
    Retrurns:
        learned_par (list): The list containing the training samples and
            the desired outputs for them.
    '''
    return [X, y]

def memorizer_apply(learned_par: list, X: np.ndarray):
    '''A function simulating Memorizer
    
         out = memorizer_apply(learned_par, X)
    
    Args:
        learned_par (list): List with training samples and the respective desired outputs.
        X (2-d numpy array of floats): The test set; samples are rows.
    
    Returns:
        out (vector of floats): whenever the i-th input vector is contained 
            within the memorized input samples, out[i] equals the i-th 
            remembered desired output, otherwise it will be randomly 0 or 1
    '''
    known = learned_par[0]
    known_out = learned_par[1]
    # generate random outputs
    out = rng.integers(0, 2, size=X.shape[0])    
    for i in range(X.shape[0]):
        for j in range(known.shape[0]):
            if np.array_equal(X[i], known[j,:]):
                out[i] = known_out[i]
                break
    return out

When applying $k$-fold cross-validation, we will compute the confidence interval for the estimation of the difference of errors computed by the `cross_val()` function. Implement the following function.

In [102]:
def conf_interval(d: float, s: float, conf_level: float, k: int) -> tuple:
    '''Compute confidence interval for the estimated difference of errors d 
    with standard deviation s returned from cross_val.
    
        low, high = conf_inteval(d, s, conf_level, k)
    
    Args:
        d: The difference of errors computed by k-fold cross-validation.
        s: The standard deviation of the difference of errors computed 
            by k-fold cross-validation.
    '''
    #I need to use t-student function for knowing the tn value
    # The coeficient multiplying the standard deviation is the value of the t-student two-sided distribution 
    # (I'm supposing it needs to be two-sided to get same values as in the power point) 
    # with parameters N (confidence level, N%) and k-1.
    #(I've tested the value x = np.abs(t.ppf((1-0.95)/2,5)) and it matches the slides value 2.57)
    from scipy.stats import t
    low = d - np.abs(t.ppf((1-(conf_level/100))/2,k-1))*s
    high = d + np.abs(t.ppf((1-(conf_level/100))/2,k-1))*s
    return low, high

The above algorithms can be compared using the above function
`cross_val`.

In [103]:
# at first we read training patterns; 
# this should be a matrix of size 600 x 2
X1 = np.genfromtxt('X1.csv', delimiter=',', dtype = float)
print(X1)
print(X1.shape)

# then we read the desired outputs for the X_traing patterns 
# (0/1 vector of length 600)
y1 = np.genfromtxt('y1.csv', delimiter=',', dtype = int)
print(y1)

# a row X1[i] is the i-th training vector with 
# the desired output y1[i].

# we prepare two lists of the learning parameters
# for the perceptron learning algorithm consisting of:
#     an extended weight vector,
#     a learning rate, and
#     a maximal number of epochs
par1 = [[1, 1, -1], 1, 10]
par2 = [[1, 1, -1], 1, 200]

# run 5-fold cross-validation
k = 5
d, s = cross_val('perc_learn', 'perc_apply', par1, 'perc_learn', 'perc_apply',
                par2, X1, y1, k, shuffle=False)

print(f"Estimated error difference {d}, standard deviation for the estimate is {s}")
# print the interval to which the true error difference belongs 
# with probability at least 95%
print(f"Confidence interval {conf_interval(d, s, 95, k)}")

[[-2.7272648  -3.61613625]
 [ 3.31729307  2.77077998]
 [ 4.74161526  3.35034658]
 ...
 [ 1.71882423 -1.07303966]
 [ 2.24315112 -0.69097537]
 [ 2.77739445  0.27689117]]
(600, 2)
[0 1 1 0 0 1 0 0 1 0 0 1 1 1 0 1 1 1 0 1 0 0 1 1 1 1 0 0 1 1 0 0 0 1 0 1 0
 1 0 1 0 0 1 1 1 1 0 1 1 1 1 0 1 1 0 1 0 1 1 1 0 1 1 0 0 1 0 0 1 1 0 0 0 1
 1 0 0 0 0 0 1 0 0 1 1 1 0 0 0 1 1 0 1 0 0 1 0 0 0 0 0 1 0 0 1 1 1 0 1 0 1
 1 0 0 1 1 0 1 0 1 0 1 0 1 0 0 0 1 1 1 1 0 0 1 1 1 0 0 0 1 1 1 0 1 0 0 0 1
 0 1 1 0 1 1 0 0 1 0 1 1 1 1 0 1 1 1 1 0 0 0 1 1 0 1 0 0 0 0 0 1 0 0 0 1 1
 1 1 1 0 1 0 1 0 0 0 0 1 1 1 0 1 0 0 1 0 1 0 1 0 1 1 0 1 1 1 1 0 0 1 0 1 1
 0 0 1 1 0 1 1 1 0 1 0 1 0 1 1 0 0 1 1 1 1 1 1 0 0 0 1 1 1 1 0 0 0 0 0 0 1
 1 1 1 0 1 0 1 1 1 0 1 0 0 1 0 0 1 0 1 1 0 1 1 0 0 1 0 0 1 1 1 1 0 0 0 0 1
 0 0 1 0 1 0 1 1 0 1 1 1 1 0 1 0 1 1 1 1 1 1 0 0 0 0 0 1 0 0 0 1 0 0 1 1 0
 1 0 1 1 1 1 1 1 1 0 0 1 0 1 1 1 0 1 1 1 1 0 1 0 1 0 0 0 0 0 0 0 0 1 0 0 1
 0 0 0 0 1 0 0 1 0 0 0 1 1 0 0 0 1 1 0 0 0 1 0 1 0 1 1 1 0 1 0 1 1 0 0 0 

**Tasks:**

1.  Implement the functions `err`, `cross_val`, `conf_interval`, and by running the above     script,
    estimate the difference in error rates of the perceptron learning
    algorithm with at most 10 epochs and of the same perceptron learning
    algorithm with at most 200 epochs. From the resulting error
    difference and the standard deviation of the estimate, compute the
    interval which contains the true error difference with the
    probability of 95%. Is the error difference statistically significant?

2.  Then modify the above script to compare error rates of the algorithm
    memorizer and perceptron with at most 50 epochs, learning rate 1.0,
    and initial extended weight vector `[1, 1, 1, 1, -1]` using 6-fold
    cross-validation on the following 600 samples:

In [104]:
# samples
X2 = np.genfromtxt('X2.csv', delimiter=',', dtype = float)

# labels for the samples
y2 = np.genfromtxt('y2.csv', delimiter=',', dtype = int)

# in order to obtain the same answers from memorizer_apply, we reset random 
# numbers generator before calling cross_val
rng = default_rng(seed=1)

In [105]:
# we prepare a list of the learning parameters
# for the perceptron learning algorithm consisting of:
#     an extended weight vector,
#     a learning rate, and
#     a maximal number of epochs
#and empty list for memorizer algorithm

par1 = [[1, 1, 1, 1, -1], 1, 50]    #parameters for percepton with the given weights, learning rate 1 and at most 50 epochs
par2 = []

# run 6-fold cross-validation
k = 6
d, s = cross_val('perc_learn', 'perc_apply', par1, 'memorizer_learn', 'memorizer_apply',
                par2, X2, y2, k, shuffle=False)
#Shuffle? I'm doing it without shuffle

print(f"Estimated error difference {d}, standard deviation for the estimate is {s}")
# print the interval to which the true error difference belongs 
# with probability at least 95%
print(f"Confidence interval {conf_interval(d, s, 95, k)}")

Estimated error difference -0.3583333333333333, standard deviation for the estimate is 0.09669252527700599
Confidence interval (-0.6068893825468165, -0.1097772841198501)


**You should submit** a Jupyter notebook with 

* commented **source code** of the functions `err`, `cross_val`, `conf_interval`, and
  all the functions you have used for solving the above tasks. Do not forget to attach your implementation of the class `Perceptron`.

* The notebook should contain **the results** of your experiments for both above tasks. You should analyze
  the obtained results, **answer the question** on the significance in both tasks. 
  
* Write **a recommendation** which
  algorithm is better to use. Eventually, you can advise which
  experiments would be suitable for a more thorough comparison of the
  considered learning algorithms.

**TASK 1**

We have the implementation in code cell **43**. We are studying the differences between the **Perceptron** algorithm with maximum numer of epochs 10 in comparison with the **Perceptron** algorithm with maximum number of epochs 200. We obtain the following results:

* Estimated error difference: 0.011666666666666665
* Standard deviation for the estimate: 0.009718253158075498
* Confidence interval: (-0.015315529745145103, 0.03864886307847844)

Is the error difference statistically significant?

The error difference estimator is 0.01, wich is smaller than the significance level value (0.05). This is not a statiscally significance error. Moreover, to analise the result we need to recall that the estimated error difference is the mean of the error differences in each k-fold step in the algorithm. These are substractions of two positive terms (because the MSE error of an algorithm is a sum of squares, therefore positive).Then, if it is positive it means that the first term (meaning the error from the first algorithm) is bigger, and if it is negative it means that the second term (meaning hte error from the second algorithm) is bigger. 

In our comparision the error difference estimator is positive. This means that the first algorithm (perceptron with at most 10 epochs) is "better" than the second algorithm (perceptron with at most 200 epochs). Therefore, we could say that the Perceptron algorithm with at most 10 epocs is better, but the difference is not statiscalli significative. BEsides that, the confidence interval low bound is negative, so we could have, with 95% a negative value for the difference and say that the second algorithm (200 epochs at most) is better. Therfore, this comparision is not significative to compare the errors between algorithms.

We could use p-values (probability of getting a more extrem estimator, level of marginal significance within a statistical hypothesis test) to talk about significancce but we havn't studied them in class.


**TASK 2**

We have the implementation in code cell **45**. We are studying the differences between the **Perceptron** algorithm with maximum numer of epochs 50, learning rate 1.0 and initial extended weight vector [1,1,1,1,-1], in comparison with the **Memorizer** algorithm (no need to specify parameters because the algorithm doesn't use any). We obtain the following results:

* Estimated error difference: -0.3583333333333333
* Standard deviation for the estimate: 0.09669252527700599
* Confidence interval: (-0.6068893825468165, -0.1097772841198501)

In this case, the error difference is statistically significant. The estimated error difference is (aprox) -0.36, which is > 0.1 (we could use this as a tolerance value, wich is actually quite big and not precise, we usually use 0.01 as tolerance value or significance level).This difference is not ignorable, and the result is clearly showing us that the error doesn't match in the compared algorithms.

The estimated error difference is the mean of the error differences in each k-fold step in the algorithm. These are substractions of two positive terms (because the MSE error of an algorithm is a sum of squares, therefore positive).Then, if it is positive it means that the first term (meaning the error from the first algorithm) is bigger, and if it is negative it means that the second term (meaning hte error from the second algorithm) is bigger.

In our comparison we obtained a negative estimator for the error difference between algorithms. This means that the secong algoritm (Memorizer) has a bigger error value than the first one. Therefore, I recomend better to use the Prceptron algorithm, instead of Memorizer.

In case we wanted to keep studying these learing algorithms to get a more thorough comparison, we could use again the k-cross fold validation with different parameters algorithms. In my opinion, could be interesting to study the following combinations:

* Perceptron algorithm with same epochs and same initial weights values, but difference learning rates.
* Reapeat some comparision that has been already done, but changing the k value to see if it effects the result.
* Compare the Perceptron that got worst error results from task 1 with Memorizer, meaning reapeat the comparison between Perceptron and Memorizer done in task 2 but changing the number of maximum epochs.
* Other comparisons to discover the effect of initial wwights value, learning rates value or maxim number of epochs in the algorithm error.


    